# CineClub Ticket Liability Analysis
**Snapshot date:** July 15, 2026

Quantifies total CineClub ticket obligations across three categories:

| Category | Description |
|---|---|
| **Current Holdings** | Tickets already credited and sitting in member accounts (`recognitions_left`) |
| **Forward Liability** | Tickets members will earn between now and their subscription end |
| **Contingent Liability** | Tickets from gift cards redeemed but not yet applied to subscriptions |

Members earn **1 free ticket per month**, starting the month after activation through to 1 month before renewal.

In [0]:
from pyspark.sql import functions as F

# Base profile: all members with unredeemed tickets as of July 5, 2026
T_profile = (
    spark.table("cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot")
    .filter((F.col("snapshot_date_id") == "20260715") & (F.col("recognitions_left") > 0))
)

# Lightweight version for downstream credit calculations
# annual_current_term_end_datetime corrects stale renewal_datetime for gift-plan members who stacked
df_T1 = T_profile.select(
    "cde_id", "cc_user_profile_id", "renewal_status", "plan_id",
    "recognitions_left", "renewal_datetime", "annual_current_term_end_datetime"
)


---
## 1 — Current Unredeemed Holdings
Counts tickets already sitting in member accounts (`recognitions_left > 0` on the snapshot).

Test/dummy accounts are excluded via a `recognitions_left < 100` filter — the highest legitimate balance for any real member is 59 tickets (CineClub launched August 2021; snapshot is July 2026).

In [0]:
# Unredeemed tickets per CDE ID
df_unredeemed = (
    df_T1
    .groupBy("cde_id")
    .agg(F.sum("recognitions_left").alias("total_recognitions_left"))
    .filter(F.col("total_recognitions_left") < 100)
)

# Total unredeemed tickets across all members
total_unredeemed = df_unredeemed.agg(F.sum("total_recognitions_left").alias("total")).collect()[0]["total"]
print(f"Total unredeemed tickets as of 2026-07-15: {total_unredeemed:,}")

display(df_unredeemed.orderBy(F.col("total_recognitions_left").desc()))

---
## 2 — Forward Ticket Liability
Projects tickets active annual and gift-plan members will earn between now and their subscription end.

### Why `effective_renewal_datetime` is needed
The snapshot `renewal_datetime` is stale for two cohorts — `annual_current_term_end_datetime` corrects this:

| Cohort | Problem | Correction |
|---|---|---|
| Gift-plan `Active_Non_Renewing` | `renewal_datetime` is in the past; a stacked gift extended the term | Use `annual_current_term_end_datetime` |
| Annual1 mid-term stackers (gap ≥ 12 months) | Gift redeemed mid-term pushed `annual_current_term_end_datetime` forward; `renewal_datetime` stayed stale | Use `annual_current_term_end_datetime` |
| Annual1 at-renewal (gap = 0) | Both dates advance together at renewal — no correction needed | Keep `renewal_datetime` as-is |

**Projection formula:** `GREATEST(0, CEIL(months_between(effective_renewal_datetime, snapshot_date) − 1))`

---
## 3 — Pending (Unstacked) Gift Cards
Loads all redeemed gift cards for annual and gift-plan members, then identifies which are still unapplied.

Members already corrected in Section 2 (`effective_renewal_datetime ≠ renewal_datetime`) are excluded — their older gifts were confirmed applied via a 100% match against `subscription_changed` events.

A gift is **pending** if `redemption_date` falls outside the 12-month window before `effective_renewal_datetime`:
* **Future** — redeemed after the current term ends; Chargebee applies it at next renewal
* **Orphaned** — redeemed more than 12 months before renewal; outside Chargebee’s auto-stack window

Returns the **full active population** — members with no pending gifts show `pending_gifts_count = 0`.

---
## 4 — Liability Summary

---
## 5 — Member-Level Output
One row per active annual/gift member.

* `gift_card_stacked = Yes` — gift applied; `effective_renewal_date` shows the corrected end date
* `gift_card_stacked = No` — no correction; `renewal_datetime` is already accurate
* `recognitions_left` — current unredeemed balance from the snapshot
* `projected_tickets_remaining` — tickets earned between now and `effective_renewal_datetime`

In [0]:
df_final_output = (
    df_active_members
    .withColumn(
        "gift_card_stacked",
        F.when(
            F.col("effective_renewal_datetime") != F.col("renewal_datetime"),
            F.lit("Yes")
        ).otherwise(F.lit("No"))
    )
    .withColumn(
        "effective_renewal_date",
        F.when(
            F.col("effective_renewal_datetime") != F.col("renewal_datetime"),
            F.to_date(F.col("effective_renewal_datetime"))
        ).otherwise(F.lit(None).cast("date"))
    )
    .select(
        "cde_id",
        "plan_id",
        F.to_date(F.col("renewal_datetime")).alias("renewal_date"),
        "gift_card_stacked",
        "effective_renewal_date",
        F.col("recognitions_left").cast("int").alias("recognitions_left"),
        "projected_tickets_remaining"
    )
    .orderBy(
        F.col("gift_card_stacked").desc(),
        F.col("projected_tickets_remaining").desc()
    )
)

stacked_count = df_final_output.filter(F.col("gift_card_stacked") == "Yes").count()
total_count   = df_final_output.count()
print(f"Total members in output : {total_count:,}")
print(f"  Gift card stacked     : {stacked_count:,}")
print(f"  No stacking           : {total_count - stacked_count:,}")

display(df_final_output)

In [0]:
# Active gift & annual members with months to renewal from July 15, 2026.
# Sources directly from the snapshot without a recognitions_left filter so that
# members who have used all current tickets but still have months remaining
# are included in the forward-looking ticket liability projection.
# For Active_Non_Renewing gift-plan members with a stale past renewal_datetime,
# annual_current_term_end_datetime reflects the true expiry after stacking.
SNAPSHOT_DATE = "2026-07-15"

df_active_members = (
    spark.table("cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot")
    .filter(
        (F.col("snapshot_date_id") == "20260715") &
        F.col("renewal_status").like("Active%") &
        (
            F.col("plan_id").ilike("%gift%") |
            F.col("plan_id").ilike("%annual%")
        )
    )
    .select(
        "cde_id", "cc_user_profile_id", "renewal_status", "plan_id",
        "recognitions_left", "renewal_datetime", "annual_current_term_end_datetime"
    )
    # Correct stale renewal_datetime using annual_current_term_end_datetime for two cohorts:
    #
    # 1. Gift-plan ANR with past renewal_datetime:
    #    The snapshot renewal_datetime is stale (past) because Chargebee extended the annual
    #    term end after a mid-term gift redemption but the snapshot didn't pick it up.
    #    annual_current_term_end_datetime > renewal_datetime confirms the extension.
    #
    # 2. Annual1 with mid-term stacked gift (gap >= 12 months):
    #    renewal_datetime correctly shows the current billing term end, but a gift stacked
    #    mid-term pushed annual_current_term_end_datetime 12+ months further out.
    #    Unlike at-renewal gifts (gap = 0, already baked into renewal_datetime), these
    #    mid-term stackers have committed future credits not captured by renewal_datetime.
    .withColumn(
        "effective_renewal_datetime",
        F.when(
            # Gift-plan ANR: renewal_datetime is in the past and stale
            (F.col("renewal_status") == "Active_Non_Renewing") &
            (F.col("plan_id").ilike("%gift%")) &
            (F.col("renewal_datetime") < F.lit(SNAPSHOT_DATE).cast("timestamp")) &
            (F.col("annual_current_term_end_datetime") > F.col("renewal_datetime")),
            F.col("annual_current_term_end_datetime")
        ).when(
            # Annual1: gift stacked mid-term, annual_current_term_end is 12+ months ahead
            F.col("plan_id").ilike("%annual%") &
            (F.col("annual_current_term_end_datetime") > F.add_months(F.col("renewal_datetime"), 11)),
            F.col("annual_current_term_end_datetime")
        ).otherwise(F.col("renewal_datetime"))
    )
    .withColumn(
        "months_from_july15",
        F.round(F.months_between(F.col("effective_renewal_datetime"), F.lit(SNAPSHOT_DATE)), 0).cast("int")
    )
    .withColumn(
        "projected_tickets_remaining",
        F.greatest(
            F.lit(0),
            F.ceil(F.months_between(F.col("effective_renewal_datetime"), F.lit(SNAPSHOT_DATE)) - 1).cast("int")
        )
    )
    .select(
        "cde_id", "cc_user_profile_id", "plan_id", "renewal_status", "recognitions_left",
        "renewal_datetime", "effective_renewal_datetime",
        "months_from_july15", "projected_tickets_remaining"
    )
)

print(f"Active gift & annual members: {df_active_members.count():,}")
display(df_active_members.orderBy("months_from_july15"))

In [0]:
# Silver layer: cc_userprofile IDs who redeemed a gift card while on annual or gift plan
df_gifting = (
    spark.table("cpx_dataanalytics_silver.cineclub.gifting")
    .filter(
        (F.col("is_redeemed") == True) &
        (
            F.col("redeemer_plan_id").ilike("%gift%") |
            F.col("redeemer_plan_id").ilike("%annual%")
        )
    )
    .select(
        F.col("redeemer_id").alias("cc_userprofile_id"),
        F.col("redeemer_plan_id").alias("plan_id"),
        F.col("redemption_date")
    )
)

print(f"Gift card redemptions by annual/gift plan members: {df_gifting.count():,}")
display(df_gifting.orderBy(F.col("redemption_date").desc()))

---
### 3b — Recent Monthly-to-Annual Upgrades
Members with `renewal_status = Active_Annual_Upgrade` on the July 15 snapshot. These are `monthly1` members who have requested an upgrade to `annual1` — the plan change takes effect on their next monthly `renewal_datetime`.

Since they are still `monthly1` on the snapshot date they are not in `df_active_members`, but their annual ticket liability is real and starts from `renewal_datetime`. `annual_renewal_date = renewal_datetime + 12 months` is used to project tickets from the snapshot date.

In [0]:
# MONTHLY-TO-ANNUAL UPGRADES
# ---------------------------------------------------------------------------
# Members with renewal_status = 'Active_Annual_Upgrade' on the July 15 snapshot.
# These monthly1 members have requested an upgrade; their annual plan starts
# on their next monthly renewal_datetime. They are not in df_active_members
# (which only includes Active% annual/gift plans) so their ticket liability
# would be missed without this cell.
#
# annual_activation_date = renewal_datetime (when monthly ends, annual starts)
# annual_renewal_date    = renewal_datetime + 12 months
# projected_tickets      = GREATEST(0, CEIL(months_between(annual_renewal_date, snapshot_date) - 1))
# ---------------------------------------------------------------------------

df_monthly_upgrades = (
    spark.table("cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot")
    .filter(
        (F.col("snapshot_date_id") == "20260715") &
        (F.col("renewal_status") == "Active_Annual_Upgrade") &
        (F.col("plan_id") == "monthly1")
    )
    .withColumn(
        "effective_renewal_datetime",
        F.add_months(F.col("renewal_datetime"), 12).cast("timestamp")
    )
    .withColumn(
        "annual_renewal_date",
        F.to_date(F.col("effective_renewal_datetime"))
    )
    .withColumn(
        "months_from_july15",
        F.round(F.months_between(F.col("effective_renewal_datetime"), F.lit(SNAPSHOT_DATE)), 0).cast("int")
    )
    .withColumn(
        "projected_tickets_remaining",
        F.greatest(
            F.lit(0),
            F.ceil(F.months_between(F.col("effective_renewal_datetime"), F.lit(SNAPSHOT_DATE)) - 1).cast("int")
        )
    )
    .select(
        "cde_id",
        "cc_user_profile_id",
        "plan_id",
        "renewal_status",
        F.col("recognitions_left").cast("int").alias("recognitions_left"),
        "renewal_datetime",
        "effective_renewal_datetime",
        F.to_date(F.col("renewal_datetime")).alias("annual_activation_date"),
        "annual_renewal_date",
        "months_from_july15",
        "projected_tickets_remaining"
    )
    .orderBy("annual_activation_date", "cde_id")
)

total         = df_monthly_upgrades.count()
total_tickets = df_monthly_upgrades.agg(F.sum("projected_tickets_remaining")).collect()[0][0] or 0

print(f"Members with pending annual upgrade      : {total:,}")
print(f"Projected tickets for this cohort        : {int(total_tickets):,}")

display(df_monthly_upgrades)

In [0]:
# PENDING (UNSTACKED) GIFT CARDS
# ---------------------------------------------------------------------------
# These are gift cards that have been redeemed but NOT yet applied to the
# member's subscription. There are two reasons a gift ends up pending:
#
#   FUTURE  : redemption_date >= effective_renewal_datetime
#             The gift arrived after the current subscription term ends.
#             Chargebee hasn't applied it yet — it will extend the subscription
#             once the current term closes.
#
#   ORPHANED: redemption_date < add_months(effective_renewal_datetime, -12)
#             The gift was redeemed more than 12 months before renewal.
#             It falls outside Chargebee's auto-stack window and sits unprocessed.
#
# For each pending member this cell calculates:
#   adjusted_renewal_datetime  — what their renewal would be once all pending
#                                gifts are applied (+12 months each)
#   extra_tickets_once_applied — incremental projected tickets from that extension
# ---------------------------------------------------------------------------

# Step 1: identify members/gifts that are genuinely pending
# Only look for pending gifts among members whose subscription end date did NOT
# need correction in Section 2. If effective_renewal_datetime != renewal_datetime,
# a prior cross-check proved those older gifts were already applied and would be
# double-counted here.
df_pending_counts = (
    df_active_members
    .filter(F.col("effective_renewal_datetime") == F.col("renewal_datetime"))
    .join(
        df_gifting.select("cc_userprofile_id", "redemption_date"),
        F.col("cc_user_profile_id") == F.col("cc_userprofile_id"),
        "inner"
    )
    .filter(
        (F.col("redemption_date") >= F.col("effective_renewal_datetime")) |
        (F.col("redemption_date") <  F.add_months(F.col("effective_renewal_datetime"), -12))
    )
    .groupBy("cc_user_profile_id")
    .agg(
        F.count("*").alias("pending_gifts_count"),
        F.max("redemption_date").alias("latest_redemption_date")
    )
)

# Step 2: return the FULL active population, with pending metrics populated only
# for members who actually have unapplied gifts. Everyone else gets zero pending gifts.
df_active_pending_stack = (
    df_active_members
    .join(df_pending_counts, "cc_user_profile_id", "left")
    .withColumn("pending_gifts_count", F.coalesce(F.col("pending_gifts_count"), F.lit(0)))
    # If all pending gifts were applied, renewal extends by 12 months per gift
    .withColumn(
        "adjusted_renewal_datetime",
        F.add_months(F.col("effective_renewal_datetime"), F.col("pending_gifts_count") * 12)
    )
    .withColumn(
        "adjusted_projected_tickets",
        F.greatest(
            F.lit(0),
            F.ceil(F.months_between(F.col("adjusted_renewal_datetime"), F.lit(SNAPSHOT_DATE)) - 1).cast("int")
        )
    )
    .withColumn(
        "extra_tickets_once_applied",
        F.col("adjusted_projected_tickets") - F.col("projected_tickets_remaining")
    )
    .withColumn(
        "has_pending_gift",
        F.when(F.col("pending_gifts_count") > 0, F.lit("Yes")).otherwise(F.lit("No"))
    )
    .select(
        "cde_id", "cc_user_profile_id", "plan_id", "renewal_status", "recognitions_left",
        "renewal_datetime", "effective_renewal_datetime",
        "months_from_july15", "projected_tickets_remaining",
        "has_pending_gift", "pending_gifts_count", "latest_redemption_date",
        "adjusted_renewal_datetime", "adjusted_projected_tickets", "extra_tickets_once_applied"
    )
)

# Step 3: same pending scan for monthly-to-annual upgrade cohort
# All upgrade members are scanned — no prior-correction exclusion applies since
# effective_renewal_datetime was set analytically (renewal_datetime + 12 months),
# not as a stacking correction.
df_upgrade_pending_counts = (
    df_monthly_upgrades
    .join(
        df_gifting.select("cc_userprofile_id", "redemption_date"),
        F.col("cc_user_profile_id") == F.col("cc_userprofile_id"),
        "inner"
    )
    .filter(
        (F.col("redemption_date") >= F.col("effective_renewal_datetime")) |
        (F.col("redemption_date") <  F.add_months(F.col("effective_renewal_datetime"), -12))
    )
    .groupBy("cc_user_profile_id")
    .agg(
        F.count("*").alias("pending_gifts_count"),
        F.max("redemption_date").alias("latest_redemption_date")
    )
)

df_upgrade_pending_stack = (
    df_monthly_upgrades
    .join(df_upgrade_pending_counts, "cc_user_profile_id", "left")
    .withColumn("pending_gifts_count", F.coalesce(F.col("pending_gifts_count"), F.lit(0)))
    .withColumn(
        "adjusted_renewal_datetime",
        F.add_months(F.col("effective_renewal_datetime"), F.col("pending_gifts_count") * 12)
    )
    .withColumn(
        "adjusted_projected_tickets",
        F.greatest(
            F.lit(0),
            F.ceil(F.months_between(F.col("adjusted_renewal_datetime"), F.lit(SNAPSHOT_DATE)) - 1).cast("int")
        )
    )
    .withColumn(
        "extra_tickets_once_applied",
        F.col("adjusted_projected_tickets") - F.col("projected_tickets_remaining")
    )
    .withColumn(
        "has_pending_gift",
        F.when(F.col("pending_gifts_count") > 0, F.lit("Yes")).otherwise(F.lit("No"))
    )
    .select(
        "cde_id", "cc_user_profile_id", "plan_id", "renewal_status", "recognitions_left",
        "renewal_datetime", "effective_renewal_datetime",
        "months_from_july15", "projected_tickets_remaining",
        "has_pending_gift", "pending_gifts_count", "latest_redemption_date",
        "adjusted_renewal_datetime", "adjusted_projected_tickets", "extra_tickets_once_applied"
    )
)

# Step 4: union active annual/gift + monthly upgrade populations into one pending-stack table
df_pending_stack = df_active_pending_stack.unionByName(df_upgrade_pending_stack)

pending_only = df_pending_stack.filter(F.col("pending_gifts_count") > 0)

totals = pending_only.agg(
    F.sum("pending_gifts_count").alias("total_pending_gifts"),
    F.sum("extra_tickets_once_applied").alias("total_extra_tickets")
).collect()[0]

total_population = df_pending_stack.count()
total_pending = pending_only.count()
print(f"Total active members in population          : {total_population:,}")
print(f"Members with pending (unstacked) gifts       : {total_pending:,}")
print(f"Total pending gifts to be applied            : {int(totals['total_pending_gifts']):,}")
print(f"Extra tickets once gifts are applied         : {int(totals['total_extra_tickets']):,}")

display(
    df_pending_stack.orderBy(
        F.col("pending_gifts_count").desc(),
        F.col("extra_tickets_once_applied").desc(),
        F.col("projected_tickets_remaining").desc()
    )
)

In [0]:
# =============================================================================
# CINECLUB TICKET LIABILITY SUMMARY — as of July 15, 2026
# =============================================================================

# --- 1. Current unredeemed balance (already in member accounts) ---
# Exclude test accounts (recognitions_left >= 100)
current_balance = (
    df_T1
    .filter(F.col("recognitions_left") < 100)
    .agg(F.sum("recognitions_left").alias("total"),
         F.count("cde_id").alias("members"))
    .collect()[0]
)

# --- 2. Forward projected tickets (all active annual/gift members) ---
# projected_tickets_remaining uses effective_renewal_datetime:
#   - gift-plan ANR stale snapshot corrected via annual_current_term_end_datetime
#   - annual1 mid-term stackers (gap >= 12m) corrected via annual_current_term_end_datetime
projected = (
    df_active_members
    .filter(F.col("projected_tickets_remaining") > 0)
    .agg(
        F.sum("projected_tickets_remaining").alias("tickets"),
        F.count("cde_id").alias("members")
    )
    .collect()[0]
)

# --- 2b. Monthly-to-annual upgrades (Active_Annual_Upgrade cohort) ---
# These monthly1 members are not in df_active_members; their annual ticket
# liability begins at their next renewal_datetime (annual activation date).
upgrade_projected = (
    df_monthly_upgrades
    .filter(F.col("projected_tickets_remaining") > 0)
    .agg(
        F.sum("projected_tickets_remaining").alias("tickets"),
        F.count("cde_id").alias("members")
    )
    .collect()[0]
)

# --- 3. Contingent: pending gifts not yet applied to subscription ---
pending = (
    df_pending_stack
    .filter(F.col("pending_gifts_count") > 0)
    .agg(
        F.count("*").alias("members"),
        F.sum("pending_gifts_count").alias("gifts"),
        F.sum("extra_tickets_once_applied").alias("extra_tickets")
    )
    .collect()[0]
)

# --- Print ---
total_forward_tickets = int(projected["tickets"]) + int(upgrade_projected["tickets"])
total_forward_members = int(projected["members"]) + int(upgrade_projected["members"])
total_committed       = int(current_balance["total"]) + total_forward_tickets
total_incl_pending    = total_committed + int(pending["extra_tickets"])

print("=" * 66)
print("   CineClub Ticket Liability Summary  |  Snapshot: July 15, 2026")
print("=" * 66)
print()
print("CURRENT HOLDINGS  (tickets already in member accounts)")
print(f"  Members with unredeemed tickets          : {int(current_balance['members']):>10,}")
print(f"  Total unredeemed tickets                 : {int(current_balance['total']):>10,}")
print()
print("FORWARD LIABILITY  (tickets earned between now and subscription end)")
print(f"  Active annual/gift members               : {int(projected['members']):>10,}")
print(f"  Projected future tickets (annual/gift)   : {int(projected['tickets']):>10,}")
print(f"  Monthly-to-annual upgrade members        : {int(upgrade_projected['members']):>10,}")
print(f"  Projected future tickets (upgrades)      : {int(upgrade_projected['tickets']):>10,}")
print(f"  ── Total forward members                 : {total_forward_members:>10,}")
print(f"  ── Total projected future tickets        : {total_forward_tickets:>10,}")
print()
print("CONTINGENT LIABILITY  (pending gifts not yet applied to subscription)")
print(f"  Members with pending gifts               : {int(pending['members']):>10,}")
print(f"  Total pending gifts                      : {int(pending['gifts']):>10,}")
print(f"  Extra tickets once applied               : {int(pending['extra_tickets']):>10,}")
print()
print("-" * 66)
print(f"  COMMITTED total (current + forward)      : {total_committed:>10,}")
print(f"  TOTAL incl. contingent                   : {total_incl_pending:>10,}")
print("=" * 66)
